# Etapa 5. Cálculo do SPI-3 e do SPI-12

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/05_spi.ipynb)

Com a distribuição escolhida na Etapa 4, calculamos o SPI de todas as estações e caracterizamos as secas e os
excessos. Para cada mês do calendário, a função `spi.calcular_spi` (em `src/spi.py`):

1. soma a chuva em 3 ou 12 meses;
2. ajusta a distribuição aos totais de todos os anos daquele mês;
3. calcula a probabilidade acumulada H(x) e a transforma no valor da normal padrão: **SPI = Φ⁻¹(H(x))**.

O capítulo 5 do caderno de aprendizagem faz essa conta à mão, para um mês só.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula:
#   1) define DE ONDE os dados serão lidos (direto do GitHub, no Colab);
#   2) baixa as funções do projeto (pasta src/ do GitHub): fdp.py, spi.py, enos.py, estatistica.py;
#   3) conecta o seu Google Drive e usa a pasta spi_oni_pr/resultados, onde ficam os resultados
#      de uma etapa para a outra.
import sys
import urllib.request
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
REPO = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/"

if Path("../dados").exists():                   # computador do orientador, dentro da pasta do projeto
    DADOS = "../dados/"
    sys.path.append("../src")
else:                                           # Colab
    DADOS = REPO + "dados/"
    Path("src").mkdir(exist_ok=True)
    for arquivo in ["fdp.py", "spi.py", "enos.py", "estatistica.py"]:
        urllib.request.urlretrieve(REPO + "src/" + arquivo, "src/" + arquivo)   # baixa o arquivo
    sys.path.append("src")

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/spi_oni_pr/resultados")
else:
    SAIDA = Path("../resultados")
SAIDA.mkdir(parents=True, exist_ok=True)
print("Dados lidos de:", DADOS)
print("Resultados gravados em:", SAIDA)

if NO_COLAB:
    !pip -q install agrometeorologiapy     # biblioteca do orientador, usada para conferir o SPI

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str, "cod_ibge": str})
mensal = pd.read_csv(DADOS + "chuva_mensal_pr.csv", dtype={"cod": str})
meso = gpd.read_file(DADOS + "pr_mesorregioes.geojson")
destaque = est[est["destaque"].notna()]           # as 4 estações em destaque
print(len(est), "estações;", len(mensal), "linhas de chuva mensal")

# Aparência padrão dos gráficos do trabalho (rode uma vez; vale para todos os gráficos do notebook)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,       # tira a borda de cima
    "axes.spines.right": False,     # tira a borda da direita
    "axes.grid": True,              # grade de fundo...
    "grid.color": "#e6e5e0",        # ...bem clara, para não competir com os dados
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,         # grade atrás das barras e linhas
})

AZUL = "#2a78d6"                    # cor única quando há só uma série
# uma cor fixa para cada estação em destaque (a cor acompanha a estação em todas as figuras)
CORES = {"Santa Helena": "#2a78d6", "Cascavel": "#eb6834", "Foz do Iguaçu": "#1baf7a", "Mercedes": "#eda100"}

from matplotlib.colors import LinearSegmentedColormap

# escala divergente: vermelho (seco / negativo) - cinza (zero) - azul (úmido / positivo)
DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#a3201f", "#e34948", "#f0efec", "#2a78d6", "#184f95"])
# cores das fases do ENOS: El Niño = águas quentes (vermelho), La Niña = águas frias (azul)
CORES_FASE = {"El Niño": "#e34948", "Neutro": "#9a9890", "La Niña": "#2a78d6"}

def mapa_base(ax):
    """Desenha os limites das mesorregiões no fundo de um mapa."""
    meso.plot(ax=ax, color="#f4f3ef", edgecolor="#9a9890", linewidth=0.6)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for lado in ["left", "bottom"]:
        ax.spines[lado].set_visible(False)

import json
from scipy import stats
import spi

## 1. A distribuição escolhida na Etapa 4

In [ ]:
arquivo = SAIDA / "fdp_escolhida.json"
if arquivo.exists():
    with open(arquivo) as f:
        escolhida = json.load(f)
else:
    escolhida = {"3": "Gama", "12": "Gama"}
    print("ATENÇÃO: fdp_escolhida.json não encontrado; rode o notebook 04. Usando Gama por enquanto.")
print("Distribuição usada:", escolhida)

## 2. SPI de todas as estações

O laço passa pelas 370 estações e calcula o SPI-3 e o SPI-12 de cada uma. O resultado (`spi_pr.csv`) vai para o
Drive: os notebooks 07 e 08 usam esse arquivo.

In [ ]:
partes = []
for cod in est["cod"]:
    serie = spi.serie_da_estacao(mensal, cod)
    s3 = spi.calcular_spi(serie, 3, escolhida["3"])
    s12 = spi.calcular_spi(serie, 12, escolhida["12"])
    tab = s3[["ano", "mes"]].copy()
    tab["cod"] = cod
    tab["spi3"] = s3["spi"].round(3)
    tab["spi12"] = s12["spi"].round(3)
    partes.append(tab)

spi_pr = pd.concat(partes)[["cod", "ano", "mes", "spi3", "spi12"]]   # junta todas as estações
spi_pr.to_csv(SAIDA / "spi_pr.csv", index=False)
print(len(spi_pr), "linhas gravadas em spi_pr.csv")
spi_pr.head(15)

## 3. Conferências

**(a) Média e desvio padrão.** Por construção, o SPI de cada estação deve ter média perto de **0** e desvio padrão
perto de **1**.

In [ ]:
resumo = spi_pr.groupby("cod")[["spi3", "spi12"]].agg(["mean", "std"])
resumo.describe().round(2)

**(b) Comparação com outra implementação.** A biblioteca `agrometeorologiapy` calcula o SPI pela distribuição Gama,
de forma independente. Em Santa Helena, com a Gama, as duas devem dar praticamente o mesmo resultado.

In [ ]:
from agrometeorologiapy.en.precipitation import spi as spi_biblioteca

sh = spi.serie_da_estacao(mensal, "2454012")
nosso = spi.calcular_spi(sh, 3, "Gama")["spi"]
datas = pd.to_datetime(dict(year=sh["ano"], month=sh["mes"], day=1))
deles = spi_biblioteca(pd.Series(sh["chuva_mm"].values, index=datas), scale=3)

diferenca = (nosso.values - deles.values)
print("Maior diferença absoluta:", round(np.nanmax(np.abs(diferenca)), 3))

**(c) Frequência das classes.** Se o SPI segue a normal padrão, cada classe de McKee tem uma frequência esperada
(por exemplo, SPI ≤ −2 em 2,3% dos meses). Compare com a frequência observada no estado.

In [ ]:
classes = ["Extremamente seco", "Severamente seco", "Moderadamente seco", "Normal",
           "Moderadamente úmido", "Muito úmido", "Extremamente úmido"]
limites = [-np.inf, -2.0, -1.5, -1.0, 1.0, 1.5, 2.0, np.inf]
esperado = [(stats.norm.cdf(limites[i + 1]) - stats.norm.cdf(limites[i])) * 100 for i in range(7)]

tabela_classes = pd.DataFrame({"esperado_%": esperado}, index=classes)
for coluna in ["spi3", "spi12"]:
    contagem = spi_pr[coluna].dropna().apply(spi.classe_spi).value_counts(normalize=True) * 100
    tabela_classes[coluna + "_%"] = contagem.reindex(classes)
tabela_classes.round(1)

## 4. Eventos de seca e de excesso (Tabela 5 do TCC)

Pela teoria das sequências (`spi.eventos`): um evento de seca começa quando o SPI-12 fica ≤ −1 e termina quando
volta acima de −1. Para cada estação em destaque, os três eventos de seca mais severos e os três de excesso.

In [ ]:
partes = []
for i, linha in destaque.iterrows():
    tab = spi_pr[spi_pr["cod"] == linha["cod"]].rename(columns={"spi12": "spi"})
    secas = spi.eventos(tab, "seca").sort_values("severidade").head(3)
    excessos = spi.eventos(tab, "excesso").sort_values("severidade", ascending=False).head(3)
    secas.insert(0, "tipo", "seca")
    excessos.insert(0, "tipo", "excesso")
    juntos = pd.concat([secas, excessos])
    juntos.insert(0, "estacao", linha["municipio"])
    partes.append(juntos)
tabela5 = pd.concat(partes)
tabela5.to_csv(SAIDA / "tabela05_eventos_spi12_destaque.csv", index=False)
tabela5

## 5. Séries do SPI (Figura 5 do TCC)

Um painel por estação em destaque. Barras vermelhas = mais seco que o normal; azuis = mais úmido. As linhas
tracejadas marcam ±1.

In [ ]:
def figura_spi(coluna, titulo, arquivo):
    fig, eixos = plt.subplots(4, 1, figsize=(12, 9), sharex=True, sharey=True)
    for ax, (i, linha) in zip(eixos, destaque.iterrows()):
        tab = spi_pr[spi_pr["cod"] == linha["cod"]].dropna(subset=[coluna])
        datas = pd.to_datetime(dict(year=tab["ano"], month=tab["mes"], day=1))
        cores = np.where(tab[coluna] < 0, "#e34948", "#2a78d6")   # vermelho se negativo, azul se positivo
        ax.bar(datas, tab[coluna], width=31, color=cores)
        ax.axhline(1, color="#9a9890", linewidth=0.8, linestyle="--")
        ax.axhline(-1, color="#9a9890", linewidth=0.8, linestyle="--")
        ax.set_title(linha["municipio"], fontsize=9, loc="left")
        ax.set_ylabel(coluna.upper().replace("SPI", "SPI-"))
    fig.suptitle(titulo)
    fig.tight_layout()
    fig.savefig(SAIDA / arquivo, dpi=300, bbox_inches="tight")
    plt.show()


figura_spi("spi3", "SPI-3 nas estações em destaque, 1981–2020", "fig05a_spi3_destaque.png")
figura_spi("spi12", "SPI-12 nas estações em destaque, 1981–2020", "fig05b_spi12_destaque.png")

## 6. O ano mais seco e o mais chuvoso no estado (Figura 6 do TCC)

O SPI-12 de dezembro resume o ano inteiro (janeiro a dezembro). A média das estações indica os anos mais secos e
mais chuvosos do Paraná; os mapas mostram como cada estação se comportou nesses dois anos.

In [ ]:
dezembro = spi_pr[(spi_pr["mes"] == 12)]
media_estado = dezembro.groupby("ano")["spi12"].mean()
ano_seco = media_estado.idxmin()
ano_umido = media_estado.idxmax()
print("Ano mais seco:", ano_seco, "| SPI-12 médio =", round(media_estado.min(), 2))
print("Ano mais chuvoso:", ano_umido, "| SPI-12 médio =", round(media_estado.max(), 2))
print()
print("Os 5 anos mais secos:", list(media_estado.sort_values().index[:5]))
print("Os 5 anos mais chuvosos:", list(media_estado.sort_values(ascending=False).index[:5]))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
for ax, ano in zip(eixos, [ano_seco, ano_umido]):
    mapa_base(ax)
    tab = dezembro[dezembro["ano"] == ano].merge(est[["cod", "lon", "lat"]], on="cod")
    pontos = ax.scatter(tab["lon"], tab["lat"], c=tab["spi12"], cmap=DIVERGENTE, vmin=-3, vmax=3, s=22,
                        edgecolor="white", linewidth=0.3)
    ax.set_title(f"SPI-12 em dezembro de {ano}", fontsize=10)
fig.colorbar(pontos, ax=eixos, label="SPI-12", shrink=0.7)
fig.savefig(SAIDA / "fig06_mapas_ano_seco_umido.png", dpi=300, bbox_inches="tight")
plt.show()

---

## O que observar

- Média ≈ 0 e desvio ≈ 1 em todas as estações: se alguma fugir disso, há algo errado na série dela.
- As frequências observadas das classes devem ficar perto das esperadas. Diferenças grandes nas classes extremas
  indicam que a distribuição não representa bem as caudas.
- Compare os anos mais secos e mais chuvosos com os anos de quebra de safra do DERAL/SEAB e com os eventos ENOS que
  você vai ver na Etapa 6.

## Exercícios

**Exercício 1.** Qual foi o evento de seca mais longo em cada estação em destaque? Em que anos?

In [ ]:
# escreva sua resposta aqui

**Exercício 2.** Use `spi.eventos` com o SPI-3 (troque `"spi12"` por `"spi3"`) em Santa Helena. Há mais eventos ou
menos que com o SPI-12? Eles duram mais ou menos? Explique por quê.

In [ ]:
# escreva sua resposta aqui

---

**Entregar.** `spi_pr.csv` (no Drive), `fig05a`, `fig05b`, `fig06`, `tabela05` e os exercícios.
**O que escrever:** item 5.4 do TCC2.